In [ ]:
from dataloading import *
from bpetokenizer import *
from model_arch import *
from RRDataset import *

In [ ]:
label_vocab, cl_train, cl_dev, cl_test, it_train, it_dev, it_test = load_rr()
tokenizer = load_bpe_tokenizer()

combined_train_data = cl_train + it_train
combined_dev_data = cl_dev + it_dev

combined_train_data = (cl_train + it_train)
combined_dev_data = (cl_dev + it_dev)

train_dataset = RRDataset(combined_train_data, tokenizer, label_vocab)
dev_dataset = RRDataset(combined_dev_data, tokenizer, label_vocab)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

In [ ]:
class Phase3ClassificationModel(nn.Module):
    def __init__(self, base_model, num_classes=13, pad_token_id=0):
        super().__init__()
        self.base_model = base_model
        self.num_classes = num_classes
        self.pad_token_id = pad_token_id
        
        self.classification_head = nn.Linear(self.base_model.cfg.d_model, num_classes)

    def forward(self, idx):
        hidden_states = self.base_model.encode(idx)
        
        mask = (idx != self.pad_token_id).float().unsqueeze(-1)
        sum_embeddings = (hidden_states * mask).sum(dim=1)
        real_token_counts = mask.sum(dim=1)
        
        pooled_output = sum_embeddings / torch.clamp(real_token_counts, min=1e-9)
        
        logits = self.classification_head(pooled_output)
        
        return logits

In [ ]:
class LoRMALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, r: int = 8, alpha: float = 8.0):
        super().__init__()
        self.base = base_layer
        self.scale = alpha / r
        
        for param in self.base.parameters():
            param.requires_grad = False
            
        d = base_layer.out_features 
        
        self.lorma_A = nn.Linear(d, r, bias=False)
        self.lorma_B = nn.Linear(r, d, bias=False)
        
        nn.init.zeros_(self.lorma_B.weight)

    def forward(self, x):
        z = self.base(x)
        return z + self.scale * self.lorma_B(self.lorma_A(z))

In [ ]:
import torch.optim

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

cfg = EncDecConfig(vocab_size=50260)
base_lm = EncoderDecoderLM(cfg)

base_lm.load_state_dict(torch.load("enc.pt", map_location=device))
for param in base_lm.parameters():
    param.requires_grad = False

rank = 8
alpha = 8.0

for name, module in base_lm.named_modules():
        if isinstance(module, (BidirectionalSelfAttention, CausalSelfAttention, CrossAttention)):
            module.q_proj = LoRMALinear(module.q_proj, rank, alpha)
            module.v_proj = LoRMALinear(module.v_proj, rank, alpha)

model = Phase3ClassificationModel(base_lm, num_classes=13, pad_token_id=0)
for name, param in model.named_parameters():
    if "lorma_" in name or "classification_head" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False
model.to(device)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=2e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
from sklearn.metrics import f1_score

epochs = 10

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    
    for batch_idx, (b_inputs, b_labels) in enumerate(train_loader):
        b_inputs, b_labels = b_inputs.to(device), b_labels.to(device)
        
        optimizer.zero_grad()
        logits = model(b_inputs)
        loss = criterion(logits, b_labels)
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
            
    model.eval()
    all_preds = []
    all_targets = []
    val_loss = 0
    
    with torch.no_grad():
        for b_inputs, b_labels in dev_loader:
            b_inputs, b_labels = b_inputs.to(device), b_labels.to(device)
            
            logits = model(b_inputs)
            loss = criterion(logits, b_labels)
            val_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(b_labels.cpu().numpy())
            
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = val_loss / len(dev_loader)
    
    print("Train Loss: ", avg_train_loss, "Val Loss:", avg_val_loss, "Val Macro-F1:", macro_f1)

torch.save(model.state_dict(), "checkpoint.pt")